# 02 - Silver Layer (Dimensional Modelling + SCD Type 2)

**Fabric analogue:** Fabric Notebook reading from Lakehouse bronze Delta tables,
writing conformed silver Delta tables used by Warehouses and Power BI.

In [1]:
import polars as pl
from deltalake import write_deltalake, DeltaTable
from pathlib import Path
from datetime import date
import shutil

BRONZE = Path('../data/bronze')
SILVER = Path('../data/silver')
SILVER.mkdir(parents=True, exist_ok=True)

def read_delta(path: Path) -> pl.DataFrame:
    return pl.from_arrow(DeltaTable(str(path)).to_pyarrow_table())

def write_silver(df: pl.DataFrame, name: str):
    target = SILVER / name
    if target.exists():
        shutil.rmtree(target)
    write_deltalake(str(target), df.to_arrow(), mode='overwrite')
    return df.height

## Silver: fact_sales_clean
- dedupe on `sale_id`
- cast `sale_ts` -> datetime, derive `sale_date`
- compute `gross_amount`
- fill null `payment_method` with 'unknown'

In [2]:
sales = read_delta(BRONZE/'bronze_sales')
print('bronze rows:', sales.height)

sales_clean = (
    sales
    .unique(subset=['sale_id'])
    .with_columns([
        pl.col('sale_ts').str.to_datetime(strict=False),
        pl.col('payment_method').fill_null('unknown'),
        (pl.col('quantity') * pl.col('unit_price')).alias('gross_amount'),
    ])
    .with_columns(pl.col('sale_ts').dt.date().alias('sale_date'))
    .drop(['_ingested_at','_source_file'])
)
write_silver(sales_clean, 'silver_fact_sales')
print('silver rows:', sales_clean.height)
sales_clean.head(5)

bronze rows: 410
silver rows: 400


sale_id,customer_id,store_id,product_sku,sale_ts,quantity,unit_price,payment_method,gross_amount,sale_date
i64,i64,str,str,datetime[μs],i64,f64,str,f64,date
20,23,"""S02""","""SKU-008""",2025-01-16 02:33:00,2,7.34,"""card""",14.68,2025-01-16
278,24,"""S01""","""SKU-014""",2025-01-19 10:51:00,1,183.24,"""card""",183.24,2025-01-19
11,78,"""S01""","""SKU-004""",2025-02-22 10:01:00,4,37.16,"""wallet""",148.64,2025-02-22
286,61,"""S01""","""SKU-009""",2025-01-24 11:28:00,2,92.89,"""cash""",185.78,2025-01-24
65,78,"""S02""","""SKU-003""",2025-01-16 05:52:00,3,173.23,"""card""",519.69,2025-01-16


## Silver: dim_products (straight conform)

In [3]:
products = read_delta(BRONZE/'bronze_products').drop(['_ingested_at','_source_file'])
write_silver(products, 'silver_dim_product')
products.head()

product_sku,product_name,category,list_price
str,str,str,f64
"""SKU-001""","""Product 1""","""Apparel""",149.5
"""SKU-002""","""Product 2""","""Home""",163.61
"""SKU-003""","""Product 3""","""Food""",153.84
"""SKU-004""","""Product 4""","""Home""",132.06
"""SKU-005""","""Product 5""","""Sports""",195.58


## Silver: dim_customer as SCD Type 2

We merge two snapshots (Jan 2025 and Apr 2025). When a customer's `city` or
`loyalty_tier` changes between snapshots we open a new row and close the old one.
Columns: `valid_from`, `valid_to`, `is_current`.

In [4]:
cj = read_delta(BRONZE/'bronze_customers_jan').drop(['_ingested_at','_source_file'])
ca = read_delta(BRONZE/'bronze_customers_apr').drop(['_ingested_at','_source_file'])

# Normalize: cast snapshot_date
cj = cj.with_columns(pl.col('snapshot_date').str.strptime(pl.Date, strict=False))
ca = ca.with_columns(pl.col('snapshot_date').str.strptime(pl.Date, strict=False))

# Union, sort by customer_id + snapshot_date
hist = pl.concat([cj, ca]).sort(['customer_id','snapshot_date'])

# Detect change relative to previous row for same customer
attrs = ['city','loyalty_tier']
hist = hist.with_columns([
    pl.col(c).shift(1).over('customer_id').alias(f'prev_{c}') for c in attrs
])
hist = hist.with_columns(
    pl.when(
        (pl.col('prev_city').is_null()) |
        (pl.col('city') != pl.col('prev_city')) |
        (pl.col('loyalty_tier') != pl.col('prev_loyalty_tier'))
    ).then(True).otherwise(False).alias('is_new_version')
)

# Keep only rows that represent a change (version boundaries)
versions = hist.filter(pl.col('is_new_version')).select(
    ['customer_id','name','city','loyalty_tier','snapshot_date']
).rename({'snapshot_date':'valid_from'})

# valid_to = next valid_from - 1 day; is_current = no successor
versions = versions.with_columns(
    pl.col('valid_from').shift(-1).over('customer_id').alias('_next_from')
)
versions = versions.with_columns([
    pl.when(pl.col('_next_from').is_not_null())
      .then(pl.col('_next_from') - pl.duration(days=1))
      .otherwise(pl.date(9999,12,31))
      .alias('valid_to'),
    pl.col('_next_from').is_null().alias('is_current'),
]).drop('_next_from')

write_silver(versions, 'silver_dim_customer_scd2')
print('SCD2 rows:', versions.height)
versions.sort(['customer_id','valid_from']).head(10)

SCD2 rows: 157


customer_id,name,city,loyalty_tier,valid_from,valid_to,is_current
i64,str,str,str,date,date,bool
1,"""Customer 1""","""Toronto""","""platinum""",2025-01-01,2025-03-31,false
1,"""Customer 1""","""Victoria""","""bronze""",2025-04-01,9999-12-31,true
2,"""Customer 2""","""Toronto""","""bronze""",2025-01-01,2025-03-31,false
2,"""Customer 2""","""Ottawa""","""bronze""",2025-04-01,9999-12-31,true
3,"""Customer 3""","""Victoria""","""bronze""",2025-01-01,2025-03-31,false
3,"""Customer 3""","""Halifax""","""silver""",2025-04-01,9999-12-31,true
4,"""Customer 4""","""Halifax""","""platinum""",2025-01-01,9999-12-31,true
5,"""Customer 5""","""Victoria""","""bronze""",2025-01-01,9999-12-31,true
6,"""Customer 6""","""Halifax""","""platinum""",2025-01-01,2025-03-31,false


### How many customers had >1 version (i.e. experienced a change)?

In [5]:
changed = (versions.group_by('customer_id').len()
           .filter(pl.col('len') > 1).height)
print(f'{changed} of {versions["customer_id"].n_unique()} customers have multiple SCD2 versions')

77 of 80 customers have multiple SCD2 versions
